In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Jatin\OneDrive\Desktop\DL projects\ecom-ai\data\reviews_clean.csv", encoding="latin1")
print(df.shape)
df.head()

(189793, 7)


,ProductName,Price,Rate,Review,Summary,clean_summary,sentiment
0,Candes 12 L Room/Personal Air Cooler?Ã¿?Ã¿(Whi...,"Â??3,999",5.0,Super!,Great cooler.. excellent air flow and for this...,great cooler excellent air flow price amazing ...,positive
1,Candes 12 L Room/Personal Air Cooler?Ã¿?Ã¿(Whi...,"Â??3,999",5.0,Awesome,Best budget 2 fit cooler. Nice cooling,best budget fit cooler nice cooling,positive
2,Candes 12 L Room/Personal Air Cooler?Ã¿?Ã¿(Whi...,"Â??3,999",3.0,Fair,The quality is good but the power of air is de...,quality good power air decent,neutral
3,Candes 12 L Room/Personal Air Cooler?Ã¿?Ã¿(Whi...,"Â??3,999",1.0,Useless product,Very bad product it's a only a fan,bad product fan,negative
4,Candes 12 L Room/Personal Air Cooler?Ã¿?Ã¿(Whi...,"Â??3,999",3.0,Fair,Ok ok product,ok ok product,neutral


In [2]:
sample_reviews = df['Summary'].dropna().head(15).tolist()
sample_ratings = df['Rate'].dropna().head(15).tolist()

for r, s in zip(sample_reviews, sample_ratings):
    print(f"Rating: {s} | Review: {r}")

Rating: 5.0 | Review: Great cooler.. excellent air flow and for this price. It's so amazing and unbelievable.Just love it ÂÂ?Â?Â
Rating: 5.0 | Review: Best budget 2 fit cooler. Nice cooling
Rating: 3.0 | Review: The quality is good but the power of air is decent
Rating: 1.0 | Review: Very bad product it's a only a fan
Rating: 3.0 | Review: Ok ok product
Rating: 5.0 | Review: The cooler is really fantastic and provides good air flow. Highly recommended
Rating: 5.0 | Review: Very good product
Rating: 3.0 | Review: Very nice
Rating: 1.0 | Review: Very bad cooler
Rating: 4.0 | Review: Very good
Rating: 5.0 | Review: Beautiful product good material and perfectly working
Rating: 5.0 | Review: Awesome
Rating: 5.0 | Review: Good
Rating: 5.0 | Review: Wonderful product, Must buy
Rating: 5.0 | Review: Nice air cooler, smart cool breeze producer


In [3]:
""" Word count < 4 AND generic word → Fake
Rating 4-5 but negative words → Mismatch → Fake
Rating 1-2 but positive words → Mismatch → Fake
Spam language — "must buy", "highly recommended", "best product" → Fake
Excessive caps ratio > 40% → Fake """

' Word count < 4 AND generic word → Fake\nRating 4-5 but negative words → Mismatch → Fake\nRating 1-2 but positive words → Mismatch → Fake\nSpam language — "must buy", "highly recommended", "best product" → Fake\nExcessive caps ratio > 40% → Fake '

In [6]:
""" def detect_fake_reviews(reviews: list, ratings: list = None) -> dict:
    
    if not reviews:
        return {"fake": 0, "real": 0, "flagged_reviews": []}

    generic_words = [
        "good", "nice", "best", "awesome", "amazing", "excellent",
        "superb", "fantastic", "wonderful", "perfect", "great",
        "must buy", "highly recommend", "very good", "very nice"
    ]
    
    specific_indicators = [
        "battery", "screen", "camera", "delivery", "quality", "price",
        "material", "size", "color", "weight", "speed", "performance",
        "air flow", "cooling", "sound", "display", "build"
    ]

    results = {"fake": 0, "real": 0, "flagged_reviews": []}

    for i, review in enumerate(reviews):
        if not isinstance(review, str) or len(review.strip()) == 0:
            continue

        review_lower = review.lower()
        word_count = len(review.split())
        reasons = []

        # Rule 1: Short + Generic + No specific detail
        is_short = word_count < 5
        is_generic = any(w in review_lower for w in generic_words)
        has_specific = any(w in review_lower for w in specific_indicators)
        
        if is_short and is_generic and not has_specific:
            reasons.append("short + generic, no detail")

        # Rule 2 & 3: Rating mismatch
        if ratings and i < len(ratings):
            rating = ratings[i]
            positive_words = ["good", "great", "nice", "best", "love", "excellent"]
            negative_words = ["bad", "worst", "terrible", "waste", "pathetic", "useless"]
            has_positive = any(w in review_lower for w in positive_words)
            has_negative = any(w in review_lower for w in negative_words)

            if rating >= 4 and has_negative:
                reasons.append("high rating but negative review")
            elif rating <= 2 and has_positive:
                reasons.append("low rating but positive review")

        # Rule 4: Spam language
        spam_phrases = ["must buy", "highly recommend", "best product", "five star"]
        if any(phrase in review_lower for phrase in spam_phrases):
            reasons.append("spam language")

        # Rule 5: Excessive caps
        caps_ratio = sum(1 for c in review if c.isupper()) / max(len(review), 1)
        if caps_ratio > 0.4:
            reasons.append("excessive caps")

        if reasons:
            results['fake'] += 1
            results['flagged_reviews'].append({
                "review": review[:100],
                "reasons": reasons
            })
        else:
            results['real'] += 1

    total = results['fake'] + results['real']
    if total > 0:
        results['fake_percent'] = round((results['fake'] / total) * 100, 2)
        results['real_percent'] = round((results['real'] / total) * 100, 2)

    return results """

' def detect_fake_reviews(reviews: list, ratings: list = None) -> dict:\n\n    if not reviews:\n        return {"fake": 0, "real": 0, "flagged_reviews": []}\n\n    generic_words = [\n        "good", "nice", "best", "awesome", "amazing", "excellent",\n        "superb", "fantastic", "wonderful", "perfect", "great",\n        "must buy", "highly recommend", "very good", "very nice"\n    ]\n\n    specific_indicators = [\n        "battery", "screen", "camera", "delivery", "quality", "price",\n        "material", "size", "color", "weight", "speed", "performance",\n        "air flow", "cooling", "sound", "display", "build"\n    ]\n\n    results = {"fake": 0, "real": 0, "flagged_reviews": []}\n\n    for i, review in enumerate(reviews):\n        if not isinstance(review, str) or len(review.strip()) == 0:\n            continue\n\n        review_lower = review.lower()\n        word_count = len(review.split())\n        reasons = []\n\n        # Rule 1: Short + Generic + No specific detail\n        

In [7]:
""" result = detect_fake_reviews(sample_reviews, sample_ratings)

print(f"Fake: {result.get('fake_percent', 0)}%")
print(f"Real: {result.get('real_percent', 0)}%")
print(f"\nFlagged Reviews:")
for r in result['flagged_reviews']:
    print(f"  Review: {r['review']}")
    print(f"  Reasons: {', '.join(r['reasons'])}")
    print() """

' result = detect_fake_reviews(sample_reviews, sample_ratings)\n\nprint(f"Fake: {result.get(\'fake_percent\', 0)}%")\nprint(f"Real: {result.get(\'real_percent\', 0)}%")\nprint(f"\nFlagged Reviews:")\nfor r in result[\'flagged_reviews\']:\n    print(f"  Review: {r[\'review\']}")\n    print(f"  Reasons: {\', \'.join(r[\'reasons\'])}")\n    print() '

In [8]:
from transformers import pipeline

sentiment_model = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    truncation=True,
    max_length=512
)

label_map = {
    'LABEL_0': 'negative',
    'LABEL_1': 'neutral',
    'LABEL_2': 'positive'
}



c:\Users\Jatin\OneDrive\Desktop\DL projects\ecom-ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 11093.56it/s]


In [9]:
for review, rating in zip(sample_reviews, sample_ratings):
    result = sentiment_model(review[:512])[0]
    predicted_sentiment = label_map[result['label']]
    confidence = round(result['score'] * 100, 2)
    
    print(f"Rating: {rating} | Sentiment: {predicted_sentiment} ({confidence}%) | Review: {review[:60]}")

Rating: 5.0 | Sentiment: positive (98.83%) | Review: Great cooler.. excellent air flow and for this price. It's s
Rating: 5.0 | Sentiment: positive (91.68%) | Review: Best budget 2 fit cooler. Nice cooling
Rating: 3.0 | Sentiment: positive (89.83%) | Review: The quality is good but the power of air is decent
Rating: 1.0 | Sentiment: negative (96.65%) | Review: Very bad product it's a only a fan
Rating: 3.0 | Sentiment: positive (47.98%) | Review: Ok ok product
Rating: 5.0 | Sentiment: positive (98.83%) | Review: The cooler is really fantastic and provides good air flow. H
Rating: 5.0 | Sentiment: positive (96.07%) | Review: Very good product
Rating: 3.0 | Sentiment: positive (95.68%) | Review: Very nice
Rating: 1.0 | Sentiment: negative (91.67%) | Review: Very bad cooler
Rating: 4.0 | Sentiment: positive (95.19%) | Review: Very good
Rating: 5.0 | Sentiment: positive (97.56%) | Review: Beautiful product good material and perfectly working
Rating: 5.0 | Sentiment: positive (90.78%) | Rev

In [12]:
for review, rating in zip(sample_reviews, sample_ratings):
    result = sentiment_model(review[:512])[0]
    sentiment = label_map[result['label']]
    confidence = round(result['score'] * 100, 2)
    
    is_fake = False
    reason = ""
    
    if rating <= 2 and sentiment == 'positive':
        is_fake = True
        reason = "low rating but positive sentiment"
    elif rating >= 4 and sentiment == 'negative':
        is_fake = True
        reason = "high rating but negative sentiment"
    elif rating == 3 and sentiment == 'positive' and confidence > 80:
        is_fake = True
        reason = "neutral rating but strongly positive sentiment"
    
    status = "FAKE" if is_fake else "REAL"
    print(f"{status} | Rating: {rating} | Sentiment: {sentiment} ({confidence}%) | Reason: {reason}")

REAL | Rating: 5.0 | Sentiment: positive (98.83%) | Reason: 
REAL | Rating: 5.0 | Sentiment: positive (91.68%) | Reason: 
FAKE | Rating: 3.0 | Sentiment: positive (89.83%) | Reason: neutral rating but strongly positive sentiment
REAL | Rating: 1.0 | Sentiment: negative (96.65%) | Reason: 
REAL | Rating: 3.0 | Sentiment: positive (47.98%) | Reason: 
REAL | Rating: 5.0 | Sentiment: positive (98.83%) | Reason: 
REAL | Rating: 5.0 | Sentiment: positive (96.07%) | Reason: 
FAKE | Rating: 3.0 | Sentiment: positive (95.68%) | Reason: neutral rating but strongly positive sentiment
REAL | Rating: 1.0 | Sentiment: negative (91.67%) | Reason: 
REAL | Rating: 4.0 | Sentiment: positive (95.19%) | Reason: 
REAL | Rating: 5.0 | Sentiment: positive (97.56%) | Reason: 
REAL | Rating: 5.0 | Sentiment: positive (90.78%) | Reason: 
REAL | Rating: 5.0 | Sentiment: positive (60.98%) | Reason: 
REAL | Rating: 5.0 | Sentiment: positive (97.2%) | Reason: 
REAL | Rating: 5.0 | Sentiment: positive (92.49%) | Rea

In [15]:
specific_indicators = [
    "battery", "screen", "camera", "delivery", "quality", "price",
    "material", "size", "color", "weight", "speed", "performance",
    "air flow", "cooling", "sound", "display", "build", "power",
    "fan", "motor", "water", "tank", "remote", "noise", "smell"
]

def detect_fake_reviews(reviews: list, ratings: list = None) -> dict:
    
    if not reviews:
        return {"fake": 0, "real": 0, "flagged_reviews": []}

    results = {"fake": 0, "real": 0, "flagged_reviews": []}

    for i, review in enumerate(reviews):
        if not isinstance(review, str) or len(review.strip()) == 0:
            continue

        review_lower = review.lower()
        has_specific = any(w in review_lower for w in specific_indicators)
        
        # Sentiment nikalo
        result = sentiment_model(review[:512])[0]
        sentiment = label_map[result['label']]
        confidence = round(result['score'] * 100, 2)
        
        is_fake = False
        reason = ""

        if ratings and i < len(ratings):
            rating = ratings[i]
            
            # Rule 1: Low rating but positive sentiment
            if rating <= 2 and sentiment == 'positive':
                is_fake = True
                reason = "low rating but positive sentiment"
            
            # Rule 2: High rating but negative sentiment
            elif rating >= 4 and sentiment == 'negative':
                is_fake = True
                reason = "high rating but negative sentiment"
            
            # Rule 3: Neutral rating but strongly positive — only if no specific detail
            elif rating == 3 and sentiment == 'positive' and confidence > 80 and not has_specific:
                is_fake = True
                reason = "neutral rating but strongly positive sentiment"

        if is_fake:
            results['fake'] += 1
            results['flagged_reviews'].append({
                "review": review[:100],
                "sentiment": sentiment,
                "confidence": confidence,
                "reason": reason
            })
        else:
            results['real'] += 1

    total = results['fake'] + results['real']
    if total > 0:
        results['fake_percent'] = round((results['fake'] / total) * 100, 2)
        results['real_percent'] = round((results['real'] / total) * 100, 2)

    return results

In [16]:
result = detect_fake_reviews(sample_reviews, sample_ratings)

print(f"Fake: {result.get('fake_percent', 0)}%")
print(f"Real: {result.get('real_percent', 0)}%")
print(f"\nFlagged Reviews:")
for r in result['flagged_reviews']:
    print(f"  Review: {r['review']}")
    print(f"  Sentiment: {r['sentiment']} ({r['confidence']}%)")
    print(f"  Reason: {r['reason']}")
    print()
    

Fake: 6.67%
Real: 93.33%

Flagged Reviews:
  Review: Very nice
  Sentiment: positive (95.68%)
  Reason: neutral rating but strongly positive sentiment

